# L5. Od obrazu do ruchu: polityki LeRobot w symulacji PushT

**Systemy mechatroniczne · Laboratorium 5 (3 h akad.)** · wykłady W12, W13

[![Otwórz w Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bpasz/pwr-systemy-mechatroniczne-dla-studenta/blob/main/laboratoria/L5-lerobot-wizja-akcja/L5_lerobot.ipynb)

**Zanim zaczniesz:** w Colab wybierz *Środowisko wykonawcze → Zmień typ środowiska wykonawczego → GPU (T4)*.
Notatnik działa też na CPU, ale wtedy automatycznie zmniejsza liczbę epizodów i kroków treningu (tylko do sprawdzenia, że wszystko działa).

**Zadanie PushT:** okrągły popychacz (sterowany pozycją 2D) ma przepchnąć klocek w kształcie litery T na zielony cel.
Obserwacja: obraz 96×96 z góry + pozycja popychacza. Akcja: docelowa pozycja popychacza (x, y). Sukces: pokrycie celu ≥ 95%.

In [ ]:
%pip install -q 'lerobot[pusht,diffusion,training]' matplotlib

> Jeśli Colab poprosi o ponowne uruchomienie środowiska po instalacji (*Restart session*), zrób to i uruchom kolejne komórki.

In [ ]:
import glob, json, os, time
import matplotlib.pyplot as plt
import numpy as np
import torch

GPU = torch.cuda.is_available()
URZADZENIE = "cuda" if GPU else "cpu"
N_EPIZODOW = 20 if GPU else 1          # epizody ewaluacji w symulatorze
KROKI = 4000 if GPU else 20            # kroki treningu własnej polityki
print("urządzenie:", URZADZENIE, "| GPU:", torch.cuda.get_device_name(0) if GPU else "brak")

## 1. Zbiór demonstracji `lerobot/pusht`

Demonstracje nagrał człowiek, sterując popychaczem myszą. Zbiór w formacie **LeRobotDataset**: tabele Parquet (stan, akcja, indeksy) + wideo MP4 z kamery.
Podgląd w przeglądarce: <https://huggingface.co/spaces/lerobot/visualize_dataset?path=%2Flerobot%2Fpusht%2Fepisode_0>.

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

ds = LeRobotDataset("lerobot/pusht")
print("epizody:", ds.meta.total_episodes, "| klatki:", ds.meta.total_frames, "| fps:", ds.fps)
for nazwa, cecha in ds.features.items():
    print(f"  {nazwa:20s} {cecha['dtype']:8s} {cecha['shape']}")
print("zadanie:", ds[0]["task"])

In [ ]:
epizody = ds.meta.episodes
start, koniec = epizody["dataset_from_index"][0], epizody["dataset_to_index"][0]
print(f"epizod 0: klatki {start}..{koniec - 1} ({(koniec - start) / ds.fps:.1f} s)")

indeksy = np.linspace(start, koniec - 1, 6).astype(int)
fig, osie = plt.subplots(1, 6, figsize=(15, 2.8))
for os_, i in zip(osie, indeksy):
    os_.imshow(ds[int(i)]["observation.image"].permute(1, 2, 0).numpy())
    os_.set_title(f"t = {(i - start) / ds.fps:.1f} s"); os_.axis("off")
plt.show()

stan = np.stack([ds[int(i)]["observation.state"].numpy() for i in range(start, koniec)])
akcja = np.stack([ds[int(i)]["action"].numpy() for i in range(start, koniec)])
plt.figure(figsize=(5, 5))
plt.plot(stan[:, 0], stan[:, 1], label="stan (pozycja popychacza)")
plt.plot(akcja[:, 0], akcja[:, 1], "--", label="akcja (pozycja zadana)")
plt.gca().invert_yaxis(); plt.axis("equal"); plt.legend(); plt.title("epizod 0")
plt.show()

**Pytania 1:**

1. Ile trwa typowy epizod? Z jaką częstotliwością polityka musi podejmować decyzje?
2. Jaka jest relacja między akcją a stanem na wykresie? Skojarz z interfejsem „pozycja zadana” z W06 i W11.
3. Dlaczego do tego zadania potrzebny jest obraz, a nie wystarczy pozycja popychacza?

## 2. Gotowa polityka Diffusion Policy

Model `lerobot/diffusion_pusht` wytrenowano (ok. 200 tys. kroków) na tym zbiorze. Został opublikowany w starszym formacie LeRobot,
więc najpierw **migrujemy** go do formatu z procesorami (normalizacja wejść i wyjść jako osobne pliki).
Oryginalny skrypt migracji w LeRobot 0.6.x ma błąd dla Diffusion Policy, poniżej wersja z poprawką.

In [ ]:
%%writefile migruj_polityke.py
"""Migracja starszych polityk LeRobot (np. lerobot/diffusion_pusht) do formatu z procesorami.

Modele opublikowane przed LeRobot 0.4 nie mają plików policy_preprocessor.json
i nowe `lerobot-eval` odmawia ich wczytania (ProcessorMigrationError). LeRobot zawiera
skrypt migracji, ale w wersji 0.6.x wywraca się on przy zapisie konfiguracji
Diffusion Policy (listy liczb z config.json, np. crop_shape, nie są zamieniane
na krotki). Ten skrypt uruchamia oryginalną migrację z tą jedną poprawką.

Użycie (Colab lub terminal):
    python migruj_polityke.py --pretrained-path lerobot/diffusion_pusht --output-dir diffusion_pusht_migrated
"""
import lerobot.processor.migrate_policy_normalization as migracja

_oryginalna = migracja.make_policy_config


def _z_krotkami(policy_type, **pola):
    pola = {
        k: tuple(v) if isinstance(v, list) and all(isinstance(x, (int, float)) for x in v) else v
        for k, v in pola.items()
    }
    return _oryginalna(policy_type, **pola)


migracja.make_policy_config = _z_krotkami

if __name__ == "__main__":
    migracja.main()

In [ ]:
!python migruj_polityke.py --pretrained-path lerobot/diffusion_pusht --output-dir diffusion_pusht_migrated
!ls diffusion_pusht_migrated

In [ ]:
def ewaluuj(sciezka_polityki, katalog, n=N_EPIZODOW):
    """Uruchamia lerobot-eval i zwraca słownik metryk."""
    partia = min(n, 10)
    os.makedirs(os.path.dirname(katalog) or ".", exist_ok=True)
    !lerobot-eval --policy.path={sciezka_polityki} --env.type=pusht --eval.n_episodes={n} \
        --eval.batch_size={partia} --policy.device={URZADZENIE} --output_dir={katalog} > {katalog}.log 2>&1
    pliki = glob.glob(f"{katalog}/**/eval_info.json", recursive=True)
    if not pliki:
        print(open(f"{katalog}.log").read()[-3000:])
        raise RuntimeError("Ewaluacja nie powiodła się, log powyżej")
    info = json.load(open(pliki[0]))
    return info["overall"] if "overall" in info else info["aggregated"]

t0 = time.time()
wynik_gotowa = ewaluuj("diffusion_pusht_migrated", "wyniki/gotowa")
print(f"czas: {time.time() - t0:.0f} s")
print({k: v for k, v in wynik_gotowa.items() if "video" not in k})

In [ ]:
from IPython.display import Video, display
filmy = sorted(glob.glob("wyniki/gotowa/**/*.mp4", recursive=True))
print(len(filmy), "filmów")
display(Video(filmy[0], embed=True, width=384))

## 3. Trening własnej polityki

Trenujemy Diffusion Policy od zera na tym samym zbiorze, ale **krótko** (domyślnie 4000 kroków na GPU, ok. 20–40 min na T4).
Wynik będzie gorszy od modelu gotowego. Porównanie pokaże, ile danych i obliczeń potrzebuje polityka uczona z demonstracji.

Podczas treningu w logu widać m.in. `loss` (funkcja straty odszumiania) i `lr` (szybkość uczenia z rozgrzewką).

In [ ]:
t0 = time.time()
!lerobot-train --dataset.repo_id=lerobot/pusht --policy.type=diffusion --env.type=pusht \
    --steps={KROKI} --batch_size=64 --log_freq=200 --save_freq={KROKI} --eval_steps=0 \
    --policy.device={URZADZENIE} --policy.push_to_hub=false --output_dir=trening/dp_wlasna 2>&1 | grep -E "step:|Checkpoint|Error"
print(f"czas treningu: {(time.time() - t0) / 60:.1f} min")

In [ ]:
wynik_wlasna = ewaluuj("trening/dp_wlasna/checkpoints/last/pretrained_model", "wyniki/wlasna")
print("gotowa :", {k: round(v, 3) for k, v in wynik_gotowa.items() if isinstance(v, (int, float))})
print("własna :", {k: round(v, 3) for k, v in wynik_wlasna.items() if isinstance(v, (int, float))})

## 4. Mini-projekt: analiza i eksperyment

1. **Porównanie.** Tabela: odsetek sukcesów (`pc_success`), średnia suma nagród, czas ewaluacji dla polityki gotowej i własnej. Obejrzyj po 3 filmy każdej polityki: jakie są typowe porażki?
2. **Eksperyment (wybierz jeden):**
   - **ACT zamiast Diffusion:** wytrenuj `--policy.type=act` z tą samą liczbą kroków. Porównaj skuteczność, czas treningu i płynność ruchu.
   - **Mniej danych:** trenuj na części epizodów (`--dataset.episodes="[0,1,2,...,49]"`). Jak skuteczność zależy od liczby demonstracji?
   - **Dłuższy trening:** podwój `KROKI`. Czy skuteczność rośnie?
3. **Wnioski (5–8 zdań):** odnieś wyniki do W13: kumulacja błędów, wielomodalność, ilość danych.
   Jakie warstwy systemu mechatronicznego (W13, tabela „Polityka uczona w systemie mechatronicznym”) byłyby potrzebne, żeby taką politykę uruchomić na prawdziwym robocie?

In [ ]:
# Miejsce na Wasz eksperyment, np.:
# !lerobot-train --dataset.repo_id=lerobot/pusht --policy.type=act --env.type=pusht \
#     --steps={KROKI} --batch_size=64 --eval_steps=0 --policy.device={URZADZENIE} \
#     --policy.push_to_hub=false --output_dir=trening/act
# wynik_act = ewaluuj("trening/act/checkpoints/last/pretrained_model", "wyniki/act")

**Źródła:** LeRobot (<https://github.com/huggingface/lerobot>, dokumentacja <https://huggingface.co/docs/lerobot>);
C. Chi i in., *Diffusion Policy*, RSS 2023; T. Z. Zhao i in., *ACT*, RSS 2023; środowisko gym-pusht (<https://github.com/huggingface/gym-pusht>).